# Tuần 09: Dữ liệu đối chiếu Hán-Việt

Mục tiêu: biến một example bank Hán-Việt thành bảng evidence, figure và analysis paragraph. Tuần này không làm NLP; ta học cách tổ chức ví dụ đối chiếu để viết paper rõ hơn.

## Trước khi chạy code

Cell setup import thư viện, kiểm tra data và tạo output folders. Bạn chỉ cần Run cell này trước.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WEEK_DIR = Path("weeks/week-09-chinese-vietnamese-contrastive-data")
DATA_PATH = WEEK_DIR / "data" / "raw" / "week09_contrastive_examples.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = "78cbedef8cdd1eb5d093864ee4e4c1c75aeb963b9537b540b49c4a251faba3a8"
if not DATA_PATH.exists():
    url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-09-chinese-vietnamese-contrastive-data/data/raw/week09_contrastive_examples.csv"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(url, DATA_PATH)
actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA ok:", actual_sha == EXPECTED_SHA)


Data file: weeks/week-09-chinese-vietnamese-contrastive-data/data/raw/week09_contrastive_examples.csv
SHA ok: True


## 1. Research loop của tuần này

```text
Chinese example + Vietnamese rendering -> phenomenon code -> pattern check -> frequency/risk table -> selected examples -> analysis paragraph
```

Một row là một cặp ví dụ. Đừng đọc nó như dữ liệu của một learner.

In [2]:
df = pd.read_csv(DATA_PATH)
df["include_in_table"] = df["include_in_table"].astype(str).str.lower().eq("true")
df["risk_score"] = pd.to_numeric(df["risk_score"], errors="coerce")
analysis_rows = df[df["include_in_table"] == True].copy()
background_rows = df[df["include_in_table"] != True].copy()
print("Raw rows:", len(df))
print("Usable contrastive examples:", len(analysis_rows))
print("Background/excluded rows:", len(background_rows))
print(analysis_rows[["example_id", "phenomenon", "chinese_example", "vietnamese_rendering", "teaching_risk"]].head().to_string(index=False))


Raw rows: 66
Usable contrastive examples: 60
Background/excluded rows: 6
example_id       phenomenon chinese_example                     vietnamese_rendering teaching_risk
      C001 time_place_order       我在学校学习汉语。            Tôi học tiếng Trung ở trường.          high
      C002 time_place_order         他明天去北京。             Ngày mai anh ấy đi Bắc Kinh.        medium
      C003 time_place_order     我们下午在图书馆见面。 Chiều nay chúng tôi gặp nhau ở thư viện.          high
      C004 time_place_order      老师在黑板上写汉字。           Thầy/cô viết chữ Hán lên bảng.          high
      C005 time_place_order      我每天早上练习声调。           Mỗi sáng tôi luyện thanh điệu.        medium


## 2. Visible pattern check với `str.contains()`

`str.contains()` trả lời câu hỏi rất nhỏ: text có chứa marker này không? Nó không tự giải thích ngữ pháp. Ta vẫn phải đọc ví dụ.

In [3]:
marker = "了"
analysis_rows["contains_marker"] = analysis_rows["chinese_example"].str.contains(marker, regex=False, na=False)
marker_examples = analysis_rows[analysis_rows["contains_marker"]]
print(f"Marker {marker!r} appears in", len(marker_examples), "usable examples")
print(marker_examples[["example_id", "phenomenon", "chinese_example", "vietnamese_rendering"]].head(8).to_string(index=False))


Marker '了' appears in 18 usable examples
example_id        phenomenon chinese_example              vietnamese_rendering
      C020     aspect_marker           我吃饭了。                   Tôi ăn cơm rồi.
      C021     aspect_marker       我学了三个月汉语。  Tôi đã học tiếng Trung ba tháng.
      C022     aspect_marker          他去了北京。            Anh ấy đã đi Bắc Kinh.
      C025     aspect_marker         她买了两张票。              Cô ấy đã mua hai vé.
      C026     aspect_marker         我们上完课了。           Chúng tôi học xong rồi.
      C028     aspect_marker     我吃饭的时候，他来了。  Khi tôi đang ăn cơm, anh ấy đến.
      C029     aspect_marker       我学汉语学了两年。 Tôi học tiếng Trung được hai năm.
      C030 result_complement          我找到票了。              Tôi tìm thấy vé rồi.


## 3. Frequency table

`value_counts()` cho biết example bank đang có nhiều ví dụ ở phenomenon nào. Đây là mô tả dataset, chưa phải kết luận về độ khó thật.

In [4]:
frequency = analysis_rows["phenomenon"].value_counts().rename_axis("phenomenon").reset_index(name="n")
frequency["percent"] = (frequency["n"] / len(analysis_rows) * 100).round(1)
frequency.to_csv(TABLE_DIR / "week09_phenomenon_frequency.csv", index=False)
print(frequency.to_string(index=False))


             phenomenon  n  percent
      result_complement 11     18.3
sino_vietnamese_lexical 11     18.3
     measure_classifier 10     16.7
          aspect_marker 10     16.7
       time_place_order  9     15.0
    tone_pinyin_mapping  9     15.0


## 4. Crosstab: phenomenon by teaching risk

`pd.crosstab()` giúp đọc risk theo từng phenomenon. High-risk nghĩa là nên ưu tiên giải thích, không phải bằng chứng learner chắc chắn sai.

In [5]:
risk_table = pd.crosstab(analysis_rows["phenomenon"], analysis_rows["teaching_risk"])
risk_table = risk_table.reindex(columns=["low", "medium", "high"], fill_value=0)
risk_table.to_csv(TABLE_DIR / "week09_pattern_by_risk.csv")
print(risk_table.to_string())


teaching_risk            low  medium  high
phenomenon                                
aspect_marker              1       3     6
measure_classifier         3       5     2
result_complement          0       5     6
sino_vietnamese_lexical    2       5     4
time_place_order           1       4     4
tone_pinyin_mapping        0       5     4


## 5. Teaching priority

Priority score là heuristic đơn giản: `n * mean_risk`. Nó giúp chọn ví dụ dạy học trước, không thay thế judgment của giáo viên.

In [6]:
priority = (
    analysis_rows.groupby("phenomenon")
    .agg(
        n=("example_id", "count"),
        mean_risk=("risk_score", "mean"),
        high_risk_examples=("teaching_risk", lambda s: int((s == "high").sum())),
    )
    .reset_index()
)
priority["mean_risk"] = priority["mean_risk"].round(2)
priority["priority_score"] = (priority["n"] * priority["mean_risk"]).round(1)
priority = priority.sort_values(["priority_score", "n"], ascending=False)
priority.to_csv(TABLE_DIR / "week09_teaching_priority_table.csv", index=False)
print(priority.to_string(index=False))


             phenomenon  n  mean_risk  high_risk_examples  priority_score
      result_complement 11       2.55                   6            28.0
          aspect_marker 10       2.50                   6            25.0
sino_vietnamese_lexical 11       2.18                   4            24.0
    tone_pinyin_mapping  9       2.44                   4            22.0
       time_place_order  9       2.33                   4            21.0
     measure_classifier 10       1.90                   2            19.0


## 6. Representative examples

Một paragraph tốt thường cần một ví dụ thật rõ. Ta chọn một high-risk example cho mỗi phenomenon, rồi dùng nó để viết implication.

In [7]:
selected_examples = (
    analysis_rows.sort_values(["risk_score", "phenomenon", "example_id"], ascending=[False, True, True])
    .groupby("phenomenon", as_index=False)
    .head(1)
    [["phenomenon", "example_id", "chinese_example", "pinyin", "vietnamese_rendering", "chinese_pattern", "vietnamese_pattern", "teaching_risk", "evidence_note"]]
    .sort_values("phenomenon")
)
selected_examples.to_csv(TABLE_DIR / "week09_selected_contrastive_examples.csv", index=False)
print(selected_examples.to_string(index=False))


             phenomenon example_id chinese_example                      pinyin          vietnamese_rendering                        chinese_pattern                         vietnamese_pattern teaching_risk                                                                                             evidence_note
          aspect_marker       C020           我吃饭了。               Wǒ chīfàn le.               Tôi ăn cơm rồi.                        verb/action + 了                          verb/action + rồi          high                                                                      Do not teach 了 as simply past tense.
     measure_classifier       C012             一张票               yì zhāng piào           một vé / một tấm vé number + flat-object classifier + noun   classifier may be omitted or lexicalized          high                               Vietnamese can omit the classifier in natural speech, so 张 needs attention.
      result_complement       C030          我找到票了。         Wǒ zhǎod

## 7. Figures cho paper

Ta xuất cả PNG và SVG. PNG dễ xem nhanh; SVG hữu ích khi cần hình sắc nét trong paper hoặc slide.

In [8]:
plt.rcParams.update({"font.size": 10, "axes.titlesize": 13, "axes.labelsize": 10})
fig, ax = plt.subplots(figsize=(8.2, 4.8))
plot_freq = frequency.sort_values("n")
ax.barh(plot_freq["phenomenon"], plot_freq["n"], color=["#2563eb", "#1f7a4d", "#b45309", "#b8325f", "#6d5bd0", "#334155"][:len(plot_freq)])
ax.set_title("Week 09 contrastive phenomena (analytic examples)")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Phenomenon")
for i, value in enumerate(plot_freq["n"]):
    ax.text(value + 0.12, i, str(value), va="center", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "week09_phenomenon_frequency.png", dpi=180)
fig.savefig(FIG_DIR / "week09_phenomenon_frequency.svg")
plt.close(fig)

heat = risk_table.loc[priority["phenomenon"]]
fig, ax = plt.subplots(figsize=(8.0, 4.8))
im = ax.imshow(heat.values, cmap="YlGnBu")
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_title("Teaching risk by contrastive phenomenon")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, int(heat.iloc[i, j]), ha="center", va="center", fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="examples")
fig.tight_layout()
fig.savefig(FIG_DIR / "week09_risk_by_phenomenon_heatmap.png", dpi=180)
fig.savefig(FIG_DIR / "week09_risk_by_phenomenon_heatmap.svg")
plt.close(fig)
print("Saved figures to", FIG_DIR)


Saved figures to weeks/week-09-chinese-vietnamese-contrastive-data/outputs/figures


## 8. Analysis paragraph

Dùng bảng + một ví dụ để viết. Giữ claim ở mức descriptive.

> Trong `60` contrastive examples dùng được, `result_complement` có priority score cao nhất. Một ví dụ đại diện là `___`, nơi Chinese dùng `___` còn Vietnamese diễn đạt bằng `___`. Pattern này nên được dạy rõ vì `___`. Vì dữ liệu synthetic và nhỏ, kết quả chỉ giúp chọn teaching examples, không chứng minh transfer.

In [9]:
top = priority.iloc[0]
example = selected_examples[selected_examples["phenomenon"] == top["phenomenon"]].iloc[0]
paragraph = (
    f"In the usable contrastive examples (N = {len(analysis_rows)}), "
    f"{top['phenomenon']} had the highest teaching-priority score ({top['priority_score']}). "
    f"Example {example['example_id']} shows '{example['chinese_example']}' alongside "
    f"'{example['vietnamese_rendering']}'. The contrast is useful for teaching because it makes "
    "the Chinese pattern visible before learners translate or produce sentences. Because the dataset is synthetic and small, "
    "the result should guide example selection rather than prove learner transfer."
)
print(paragraph)


In the usable contrastive examples (N = 60), result_complement had the highest teaching-priority score (28.0). Example C030 shows '我找到票了。' alongside 'Tôi tìm thấy vé rồi.'. The contrast is useful for teaching because it makes the Chinese pattern visible before learners translate or produce sentences. Because the dataset is synthetic and small, the result should guide example selection rather than prove learner transfer.
